# 21.5 — PCA Analysis with SVM on Iris / 鸢尾花上的PCA和SVM分析

**Chapter 21 — File 5 of 5 / 第21章 — 第5个文件（共5个）**

## Summary / 总结

Apply PCA to the Iris dataset and compare classification performance using all features vs. using only principal components. Demonstrates how PCA reduces dimensionality while preserving classification ability.

对鸢尾花数据集应用PCA，比较使用所有特征与仅使用主成分的分类性能。演示PCA如何在保留分类能力的同时降低维度。

## Workflow / 工作流程

1. Load Iris data
2. Perform PCA to extract principal components
3. Visualize what each component captures
4. Remove components progressively
5. Compare SVM accuracy with all features vs. PC1 only

## Step 1 — Import Libraries / 导入库

In [ ]:
# Import all required libraries
# 导入所有必需的库
from sklearn.datasets import load_iris
from sklearn.model_selection import train_test_split
from sklearn.decomposition import PCA
from sklearn.metrics import f1_score
from sklearn.svm import SVC
import matplotlib.pyplot as plt
import numpy as np

## Step 2 — Load and Prepare Data / 加载和准备数据

In [ ]:
# Load iris dataset
# 加载鸢尾花数据集
irisdata = load_iris()
X, y = irisdata['data'], irisdata['target']
print(f"Original data shape: {X.shape}")
print(f"Classes: {irisdata['target_names']}")

## Step 3 — Visualize Raw Data / 可视化原始数据

In [ ]:
# Plot two features to show raw data structure
# 绘制两个特征以显示原始数据结构
plt.figure(figsize=(8, 6))
for i, name in enumerate(irisdata['target_names']):
    indices = y == i
    plt.scatter(X[indices, 0], X[indices, 1], label=name, alpha=0.7)
plt.xlabel(irisdata['feature_names'][0])
plt.ylabel(irisdata['feature_names'][1])
plt.title('Two Features from the Iris Dataset')
plt.legend()
plt.show()

## Step 4 — Fit PCA and Analyze Components / 拟合PCA并分析成分

In [ ]:
# Perform PCA on full data (keeping all 4 components)
# 对全数据执行PCA（保留全部4个成分）
pca = PCA().fit(X)

print("Principal Components:")
print(pca.components_)

print(f"\nExplained Variance Ratio:")
for i, ratio in enumerate(pca.explained_variance_ratio_):
    print(f"  PC{i+1}: {ratio:.4f}")

print(f"\nCumulative Explained Variance:")
cumsum = 0
for i, ratio in enumerate(pca.explained_variance_ratio_):
    cumsum += ratio
    print(f"  PC{i+1}: {cumsum:.4f}")

## Step 5 — Visualize Component Importance / 可视化成分重要性

In [ ]:
# Plot explained variance
# 绘制解释方差
plt.figure(figsize=(10, 5))

# Plot 1: Variance explained by each component
plt.subplot(1, 2, 1)
plt.bar(range(1, 5), pca.explained_variance_ratio_)
plt.xlabel('Principal Component')
plt.ylabel('Explained Variance Ratio')
plt.title('Variance Explained by Each Component')
plt.xticks(range(1, 5))

# Plot 2: Cumulative variance
plt.subplot(1, 2, 2)
cumsum = np.cumsum(pca.explained_variance_ratio_)
plt.plot(range(1, 5), cumsum, 'b-o')
plt.xlabel('Number of Components')
plt.ylabel('Cumulative Explained Variance')
plt.title('Cumulative Explained Variance')
plt.axhline(y=0.95, color='r', linestyle='--', label='95% threshold')
plt.xticks(range(1, 5))
plt.legend()
plt.tight_layout()
plt.show()

## Step 6 — Analyze Feature Contributions / 分析特征贡献

In [ ]:
# Show how original features contribute to principal components
# 显示原始特征如何贡献给主成分
print("Feature contributions to first 2 principal components:")
print(f"\nPC1 loadings (importance of each feature):")
for i, (feature, loading) in enumerate(zip(irisdata['feature_names'], pca.components_[0])):
    print(f"  {feature}: {loading:.4f}")

print(f"\nPC2 loadings:")
for i, (feature, loading) in enumerate(zip(irisdata['feature_names'], pca.components_[1])):
    print(f"  {feature}: {loading:.4f}")

## Step 7 — Transform Data and Visualize / 变换数据并可视化

In [ ]:
# Transform to PC space and visualize
# 变换到PC空间并可视化
X_pca = pca.transform(X)

plt.figure(figsize=(8, 6))
for i, name in enumerate(irisdata['target_names']):
    indices = y == i
    plt.scatter(X_pca[indices, 0], X_pca[indices, 1], label=name, alpha=0.7)
plt.xlabel(f'PC1 ({pca.explained_variance_ratio_[0]:.1%})')
plt.ylabel(f'PC2 ({pca.explained_variance_ratio_[1]:.1%})')
plt.title('Iris Dataset in Principal Component Space')
plt.legend()
plt.grid(True, alpha=0.3)
plt.show()

## Step 8 — Train/Test Split / 训练/测试分割

In [ ]:
# Split data for classification
# 分割数据用于分类
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.33, random_state=42)
print(f"Training set size: {X_train.shape[0]}")
print(f"Test set size: {X_test.shape[0]}")

## Step 9 — Compare Classifiers / 比较分类器

In [ ]:
# Train SVM with all features
# 用所有特征训练SVM
clf_all = SVC(kernel='linear', gamma='auto').fit(X_train, y_train)
y_pred_all = clf_all.predict(X_test)
acc_all = clf_all.score(X_test, y_test)
f1_all = f1_score(y_test, y_pred_all, average='macro')

print("Using all 4 features:")
print(f"  Accuracy: {acc_all:.4f}")
print(f"  F1 Score: {f1_all:.4f}")

In [ ]:
# Train SVM with only PC1
# 仅用PC1训练SVM
X_train_pc1 = X_train @ pca.components_[0].reshape(-1, 1)
X_test_pc1 = X_test @ pca.components_[0].reshape(-1, 1)

clf_pc1 = SVC(kernel='linear', gamma='auto').fit(X_train_pc1, y_train)
y_pred_pc1 = clf_pc1.predict(X_test_pc1)
acc_pc1 = clf_pc1.score(X_test_pc1, y_test)
f1_pc1 = f1_score(y_test, y_pred_pc1, average='macro')

print(f"\nUsing only PC1 ({pca.explained_variance_ratio_[0]:.1%} variance):")
print(f"  Accuracy: {acc_pc1:.4f}")
print(f"  F1 Score: {f1_pc1:.4f}")

## Step 10 — Summary / 总结

In [ ]:
# Print comparison summary
# 打印比较总结
print(f"\nComparison Summary:")
print(f"  All features (4D): Accuracy = {acc_all:.4f}")
print(f"  PC1 only (1D):    Accuracy = {acc_pc1:.4f}")
print(f"  Accuracy loss:     {acc_all - acc_pc1:.4f}")
print(f"\nConclusion: PC1 alone captures enough information for reasonable classification")
print(f"despite explaining only {pca.explained_variance_ratio_[0]:.1%} of total variance")

## Learning Notes / 学习笔记

- **Math Essence**: PCA extracts linear combinations of original features that maximize variance. These combinations (principal components) can have higher discriminative power for classification than individual original features, even though they explain less individual variance.
  
  **数学本质**：PCA提取最大化方差的原始特征的线性组合。这些组合可能对分类具有比单个原始特征更高的区分能力。

- **ML Application**: (1) Use PCA for dimensionality reduction before classification to reduce overfitting and training time, (2) The number of components should be chosen based on downstream task requirements, not just variance threshold, (3) For Iris, even 1 component achieves good classification, but 2-3 components are preferred for better generalization and understanding data structure.
  
  **ML应用**：(1) 在分类前使用PCA进行降维以减少过拟合和训练时间，(2) 成分数应基于下游任务要求选择，(3) 对于鸢尾花，甚至1个成分也能实现良好分类。

➡️ **Next / 下一步**: `../chapter_22/01_download_data.ipynb` — Country comparison using World Bank data

## Complete Code / 完整代码一览

In [ ]:
# --- Import Section / 导入部分 ---
from sklearn.datasets import load_iris
from sklearn.model_selection import train_test_split
from sklearn.decomposition import PCA
from sklearn.metrics import f1_score
from sklearn.svm import SVC
import matplotlib.pyplot as plt

# --- Load Data / 加载数据 ---
irisdata = load_iris()
X, y = irisdata['data'], irisdata['target']

# --- PCA Analysis / PCA分析 ---
pca = PCA().fit(X)
print("Principal components:")
print(pca.components_)
print("Explained variance:")
print(pca.explained_variance_)

# --- Visualization / 可视化 ---
X_pca = pca.transform(X)
plt.scatter(X_pca[:, 0], X_pca[:, 1], c=y)
plt.show()

# --- Classification Comparison / 分类比较 ---
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.33)
clf = SVC(kernel='linear', gamma='auto').fit(X_train, y_train)
print(f"Accuracy (all features): {clf.score(X_test, y_test)}")